# 06 MODE_RATE: onset rates by operating mode

Onsets per operating hour by mode for June 2025, as a cross-check of Table 5.

Inputs: `build/fault_episodes_v4.csv`, `build/telemetry_labelled_v4.pkl`.

Paper: Table 5.

In [ ]:
import pandas as pd

df = pd.read_pickle("build/telemetry_labelled_v4.pkl")
ep = pd.read_csv("build/fault_episodes_v4.csv", parse_dates=["onset"])

june = df[(df.DATETIME >= "2025-06-01") & (df.DATETIME < "2025-07-01")]
june_ep = ep[(ep.onset >= "2025-06-01") & (ep.onset < "2025-07-01")]

# operating hours per mode: one row per second
hours = june.groupby("OP").size() / 3600

# mode at each onset second, matched within one second
op = df.set_index("DATETIME")["OP"].sort_index()
mode_at_onset = op.reindex(june_ep["onset"], method="nearest", tolerance=pd.Timedelta("1s"))
onsets = mode_at_onset.value_counts()

out = pd.concat([hours.rename("hours"),
                 onsets.rename("onsets")], axis=1).fillna(0)
out["faults_per_hour"] = out["onsets"] / out["hours"]
print(out.round(2).to_string())
print(f"\nunmatched onsets: {mode_at_onset.isna().sum()} of {len(june_ep)}")

___
**END**